# 02 — Feature Engineering

**Fish Habitat / PFZ — Problem Statement B**

Two jobs here, and the first is far more consequential than the second:

1. **Construct the labels.** OBIS is presence-only, so the negative class does
   not exist and must be built. How it is built determines what the model learns.
2. **Derive habitat features** on top of the shared ocean state — fronts,
   prey-availability lags, thermal niche, bathymetric context.

Everything is written so that **fitting and applying are separate calls**
(`fit_thermal_niche` / `apply_thermal_niche`). That is not ceremony: a niche
fitted on all the data leaks held-out conditions into a training feature, and
the resulting model looks better than it is.

In [ ]:
import sys, warnings
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
sys.path.insert(0, str(PROJECT_ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from marine_ml import config, fusion, viz
from marine_ml.sources import copernicus, gebco, obis
from fish_habitat_prediction.src import labels as label_lib
from fish_habitat_prediction.src import features as feature_lib

viz.use_house_style()

REGION = config.NORTH_INDIAN_OCEAN
START, END = config.HABITAT_START, config.HABITAT_END

physics = copernicus.fetch_physics(REGION, START, END, cadence="monthly")
bgc = copernicus.fetch_bgc(REGION, START, END, cadence="monthly")
bathymetry = gebco.fetch_bathymetry(REGION)
presences = obis.fetch_all_target_species(None, REGION, START, END)
target_group = obis.fetch_target_group(REGION, START, END)

SPECIES_COLOR = viz.species_colors(config.TARGET_SPECIES.keys())
print(f"{len(presences)} presences · {len(target_group)} background-pool records")

## 1. Spatial thinning

Ten trawl records from one survey, in one bay, on one afternoon are **ten copies
of a single environmental observation**. Left in, that cell's conditions get
weighted ten times over and the model reads the survey's itinerary as the
species' preference.

`thin_presences` keeps at most one record per grid cell per month per species,
choosing the survivor at random rather than taking the first — first-seen would
bias toward whichever dataset happens to sort first.

In [ ]:
thinned = label_lib.thin_presences(
    presences, resolution=config.GRID_RESOLUTION, period="M", seed=config.RANDOM_SEED
)

summary = pd.DataFrame({
    "raw": presences.groupby("species_key").size(),
    "thinned": thinned.groupby("species_key").size(),
}).fillna(0).astype(int)
summary["removed"] = summary["raw"] - summary["thinned"]
summary["retained_%"] = (100 * summary["thinned"] / summary["raw"]).round(0)
print(f"total: {len(presences)} → {len(thinned)} ({len(thinned)/len(presences):.0%} retained)\n")
summary

## 2. Pseudo-absence construction — the decision that matters

The EDA established that occurrence records cluster around survey effort. Three
options were on the table:

| Approach | What the model would learn |
|---|---|
| Random ocean points | "Has anyone ever sampled here?" — the sampling map, not the habitat |
| Fishing effort (AIS) as negatives | Circular: effort follows fish, so the model reproduces existing fishing patterns |
| **Target-group background** | Environmental difference between where *this* species was found and where *comparable surveys* found something else |

We use the third. Background points come from Actinopterygii records — same
surveys, same gear, same places — with the focal species excluded, and with the
exact (cell, month) slots its own presences occupy removed so a background point
never contradicts a presence.

In [ ]:
training_table = label_lib.build_training_table(
    presences, target_group,
    species=config.TARGET_SPECIES,
    ratio=3.0,                       # 3 background points per presence
    resolution=config.GRID_RESOLUTION,
    seed=config.RANDOM_SEED,
)
balance = label_lib.summarise(training_table)
print(f"{len(training_table)} labelled rows "
      f"({int(training_table.presence.sum())} presence / "
      f"{int((1 - training_table.presence).sum())} background)\n")
balance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.4))

# --- class balance -------------------------------------------------------
keys = balance.species_key.tolist()
y = np.arange(len(keys))
axes[0].barh(y, balance.background, color=viz.BACKGROUND, label="background")
axes[0].barh(y, balance.presence, color=viz.PRESENCE, label="presence")
axes[0].set_yticks(y)
axes[0].set_yticklabels([k.replace("_", " ") for k in keys])
axes[0].grid(axis="x"); axes[0].grid(axis="y", visible=False)
axes[0].legend(loc="lower right")
viz.label_axes(axes[0], title="Class balance by species",
               subtitle="Roughly 3:1 background to presence, by construction.",
               xlabel="records")

# --- do presences and background occupy the same space? ------------------
pres = training_table[training_table.presence == 1]
back = training_table[training_table.presence == 0]
viz.annotate_land(axes[1], bathymetry, REGION)
axes[1].scatter(back.longitude, back.latitude, s=6, alpha=0.3,
                color=viz.BACKGROUND, edgecolor="none", label="background")
axes[1].scatter(pres.longitude, pres.latitude, s=10, alpha=0.6,
                color=viz.PRESENCE, edgecolor="none", label="presence")
axes[1].legend(loc="lower left", markerscale=2)
viz.label_axes(axes[1], title="Presence vs background in space",
               subtitle="Overlapping footprints are the goal — that is the sampling bias cancelling.",
               xlabel="longitude (°E)", ylabel="latitude (°N)")
plt.tight_layout()
plt.show()

The overlap in the right panel is the whole point. If the grey points covered
the ocean uniformly while the blue points clustered, the model's easiest win
would be geography rather than ecology.

## 3. Sampling ocean state at the labelled points

`fusion.sample_at_points` snaps each record to the common grid and reads the
environment there.

One ordering detail matters: **derived fields are computed on the grid *before*
sampling**. A spatial gradient or an Okubo–Weiss parameter is a property of a
field's neighbourhood and simply cannot be recovered from an isolated point, so
computing them after sampling would silently produce a different (wrong) feature
under the same name.

In [ ]:
import time
t0 = time.time()
features = feature_lib.build_features(
    training_table, physics, bgc, bathymetry,
    region=REGION, resolution=config.GRID_RESOLUTION,
)
print(f"built {features.shape[0]} rows × {features.shape[1]} columns in {time.time()-t0:.1f}s")
features.head(3)

### What `build_features` added, and why

| Feature group | Columns | Rationale |
|---|---|---|
| **Ocean state** | `thetao`, `so`, `uo`, `vo`, `zos`, `mlotst`, `chl`, `no3`, `po4`, `si`, `o2`, `nppv` | the physical/biogeochemical environment |
| **Fronts** | `sst_gradient`, `chl_gradient` | the learned version of INCOIS's operational SST-front heuristic |
| **Eddies** | `okubo_weiss`, `relative_vorticity`, `eddy_kinetic_energy`, `current_speed` | many pelagic species aggregate at eddy edges |
| **Prey lag** | `chl_lag30`, `chl_lag60`, `nppv_lag30/60`, `chl_trend_30d` | a bloom takes weeks to propagate up the food chain — concurrent chlorophyll is the wrong question |
| **Bathymetry** | `depth`, `log_depth`, `depth_bucket`, `seafloor_slope`, `distance_to_coast` | shelf breaks and slopes concentrate fish |
| **Season** | `doy_sin`, `doy_cos`, `month_sin`, `month_cos`, `monsoon_phase` | cyclical encoding avoids a false 31-Dec/1-Jan discontinuity |

In [ ]:
# Prey lag: is lagged chlorophyll actually different from concurrent?
lag_cols = [c for c in features.columns if c.startswith("chl_lag") or c == "chl"]
corr = features[lag_cols].corr(method="spearman")
print("Correlation between concurrent and lagged chlorophyll:")
print(corr["chl"].round(3).to_string())
print()
print("Correlations well below 1.0 mean the lag carries information the")
print("concurrent value does not — which is the justification for including it.")

## 4. Thermal niche — fit and apply, separately

The thermal-niche features place a record's local temperature within its
species' observed thermal range. That range has to be **fitted on training
presences only**.

Here we demonstrate the fit on the whole dataset for inspection, but the
training pipeline (notebook `03`) refits it *inside every cross-validation fold*
on that fold's training presences. Fitting once outside the loop would leak
held-out temperatures into a training feature and inflate every fold.

In [ ]:
niche = feature_lib.fit_thermal_niche(features)
niche_display = niche.copy()
niche_display.columns = ["species", "5th pct (°C)", "95th pct (°C)", "median (°C)"]
niche_display.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

for i, row in niche.iterrows():
    key = row.species_key
    color = SPECIES_COLOR[key]
    # Range bar plus a marker at the optimum: two encodings, one entity.
    ax.plot([row.thermal_low, row.thermal_high], [i, i],
            color=color, linewidth=6, solid_capstyle="round", alpha=0.85)
    ax.plot(row.thermal_optimum, i, "o", color=viz.SURFACE,
            markersize=9, markeredgecolor=color, markeredgewidth=2.5, zorder=3)

ax.set_yticks(range(len(niche)))
ax.set_yticklabels([k.replace("_", " ") for k in niche.species_key])
ax.grid(axis="x"); ax.grid(axis="y", visible=False)
viz.label_axes(ax, title="Thermal niche by species",
               subtitle="Bar = 5th–95th percentile of occupied SST; ring = median. The ranges overlap heavily.",
               xlabel="sea surface temperature (°C)")
plt.tight_layout()
plt.show()

The heavy overlap confirms the EDA finding: **temperature is a weak
discriminator between these species within this region.** They all occupy warm
tropical water. The thermal-niche feature is still worth having — it tells the
model when conditions are outside a species' normal range — but it will not be
what separates the species from one another.

In [ ]:
featured = feature_lib.apply_thermal_niche(features, niche)
print("added:", [c for c in featured.columns if c not in features.columns])
print()
print(featured[["species_key", "thetao", "thermal_position",
                "thermal_distance", "in_thermal_range"]].head(8).round(3).to_string(index=False))
print()
print(f"records inside their species' thermal range: {featured.in_thermal_range.mean():.1%}")
print("(≈90% by construction — the range is the 5th–95th percentile)")

## 5. Feature inventory and data quality

Before modelling, check what fraction of each feature is actually observed. A
column that is mostly missing is a column the model will mostly ignore — or
worse, one whose missingness pattern is itself predictive of something spurious.

In [ ]:
model_columns = feature_lib.feature_columns(featured)
print(f"{len(model_columns)} model input columns")
print()

missing = (
    featured[model_columns]
    .isna().mean()
    .sort_values(ascending=False)
    .rename("missing_fraction")
    .to_frame()
)
missing["dtype"] = [str(featured[c].dtype) for c in missing.index]
print("Most-missing features:")
missing.head(12).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
coverage = (1 - featured[model_columns].isna().mean()).sort_values()

# Single series → one colour. Emphasis marks the ones that fall short.
threshold = 0.9
colors = [viz.ACCENT if c < threshold else viz.PRESENCE for c in coverage.values]
ax.bar(range(len(coverage)), coverage.values, color=colors, width=0.85)
ax.axhline(threshold, color=viz.INK_MUTED, linewidth=1)
ax.text(len(coverage) * 0.99, threshold + 0.012, "90% coverage",
        ha="right", fontsize=8, color=viz.INK_SECONDARY)
ax.set_xticks([])
ax.set_ylim(0, 1.02)
viz.label_axes(ax, title="Feature coverage",
               subtitle=f"{(coverage < threshold).sum()} of {len(coverage)} features fall below 90% — mostly the eddy and gradient fields near coastlines.",
               xlabel="features (sorted)", ylabel="fraction observed")
plt.tight_layout()
plt.show()

print("Lowest-coverage features:")
print(coverage.head(6).round(3).to_string())

The gradient and eddy fields are the least complete, which is expected: they are
computed by differencing neighbouring cells, so a cell adjacent to land has no
valid neighbour on one side. This is genuine missingness, not a bug — and
LightGBM handles it natively rather than requiring imputation.

In [ ]:
usable = feature_lib.drop_unusable_rows(featured, model_columns, min_coverage=0.5)
print(f"{len(featured)} → {len(usable)} rows "
      f"({len(usable)/len(featured):.1%} retained after dropping mostly-empty rows)")
print()
print("Class balance after cleaning:")
print(usable.groupby(["species_key", "presence"]).size().unstack(fill_value=0)
      .rename(columns={0: "background", 1: "presence"}).to_string())

## 6. Which features separate the classes?

A quick univariate check before modelling — not feature selection, just a
sanity read on whether anything discriminates at all. If nothing did, the
problem would be in the labels rather than the model.

In [ ]:
from scipy import stats

rows = []
numeric_columns = [c for c in model_columns
                   if pd.api.types.is_numeric_dtype(usable[c])]
for column in tqdm(numeric_columns, desc="univariate separation"):
    present = usable.loc[usable.presence == 1, column].dropna()
    absent = usable.loc[usable.presence == 0, column].dropna()
    if len(present) < 30 or len(absent) < 30:
        continue
    # AUC via the Mann-Whitney U statistic: P(random presence > random background).
    u_stat, p_value = stats.mannwhitneyu(present, absent, alternative="two-sided")
    auc = u_stat / (len(present) * len(absent))
    rows.append({"feature": column, "auc": auc,
                 "separation": abs(auc - 0.5) * 2, "p_value": p_value})

separation = pd.DataFrame(rows).sort_values("separation", ascending=False)
separation.head(15).round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
top = separation.head(18).iloc[::-1]

# Diverging around the 0.5 no-information point: direction is meaningful
# (does the feature go up or down with presence?), so a two-pole ramp is right.
colors = [viz.CATEGORICAL[0] if a > 0.5 else viz.CATEGORICAL[7] for a in top.auc]
ax.barh(range(len(top)), top.auc - 0.5, left=0.5, color=colors, height=0.72)
ax.axvline(0.5, color=viz.INK_MUTED, linewidth=1)
ax.set_yticks(range(len(top)))
ax.set_yticklabels(top.feature, fontsize=8)
ax.grid(axis="x"); ax.grid(axis="y", visible=False)
ax.set_xlim(0.3, 0.8)
viz.label_axes(ax, title="Univariate separation (presence vs background)",
               subtitle="Blue = higher at presences, red = lower. 0.5 means no information.",
               xlabel="AUC of a single feature")
plt.tight_layout()
plt.show()

As the EDA predicted, **bathymetric and coastal features lead**. Note this is a
*univariate* view — it says nothing about which features contribute after the
others are accounted for. That is what SHAP in notebook `04` answers.

## 7. Persist to the feature store

Written to the shared feature store so notebook `03` starts from an identical
table rather than rebuilding it — and so training and any future serving path
read the same feature definitions.

In [ ]:
FEATURE_STORE_NAME = "fish_habitat_points"
fusion.write_feature_store(featured, FEATURE_STORE_NAME)

path = config.FEATURE_STORE_DIR / f"{FEATURE_STORE_NAME}.parquet"
print(f"wrote {path}")
print(f"  {featured.shape[0]} rows × {featured.shape[1]} columns")
print(f"  {path.stat().st_size / 1e6:.1f} MB")
print()
print("Verifying round-trip:")
reloaded = fusion.read_feature_store(FEATURE_STORE_NAME)
print(f"  read back {reloaded.shape[0]} rows × {reloaded.shape[1]} columns")
print(f"  presence count matches: {int(reloaded.presence.sum()) == int(featured.presence.sum())}")

---

## Summary

- Presences are **spatially thinned** (one per grid cell per month per species),
  then paired with target-group background at roughly 3:1.
- **Pseudo-absences are target-group background**, not random ocean points —
  the single most important choice in this problem.
- Features span ocean state, fronts, eddies, lagged prey availability,
  bathymetric context and cyclical season.
- **Raw latitude/longitude are excluded** from model inputs (see
  `features.EXCLUDED`): with clustered presences a tree model would memorise
  coordinates and score well while learning nothing transferable.
- Thermal niche is **fit-then-apply**, and notebook `03` refits it per fold.

Next: **`03_model_training`** — three model tiers under spatial block
cross-validation, which is the only way to find out whether any of this
generalises to water the model has not seen.